<a href="https://colab.research.google.com/github/troyam/2015/blob/master/avatarify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Avatarify Colab Server

This Colab notebook is for running Avatarify rendering server. It allows you to run Avatarify on your computer **without GPU** in this way:

1. When this notebook is executed, it starts listening for incoming requests from your computer;
1. You start the client on your computer and it connects to the notebook and starts sending requests;
1. This notebooks receives the requests from your computer, renders avatar images and sends them back;

To this end, all the heavy work is offloaded from your computer to this notebook so you don't need to have a beafy hardware on your PC anymore.


## Start the server
Run the cells below (Shift+Enter) sequentially and pay attention to the hints and instructions included in this notebook.

At the end you will get a command for running the client on your computer.

## Start the client

Make sure you have installed the latest version of Avatarify on your computer. Refer to the [README](https://github.com/alievk/avatarify#install) for the instructions.

When it's ready execute this notebook and get the command for running the client on your computer.


### Technical details

The client on your computer connects to the server via `ngrok` TCP tunnel or a reverse `ssh` tunnel.

`ngrok`, while easy to use, can induce a considerable network lag ranging from dozens of milliseconds to a second. This can lead to a poor experience.

A more stable connection could be established using a reverse `ssh` tunnel to a host with a public IP, like an AWS `t3.micro` (free) instance. This notebook provides a script for creating a tunnel, but launching an instance in a cloud is on your own (find the manual below).

# Install

### Avatarify
Follow the steps below to clone Avatarify and install the dependencies.

In [72]:
!cd /content
!rm -rf *

In [73]:
!git clone https://github.com/alievk/avatarify.git

Cloning into 'avatarify'...
remote: Enumerating objects: 1514, done.
remote: Total 1514 (delta 0), reused 0 (delta 0), pack-reused 1514 (from 1)
Receiving objects: 100% (1514/1514), 5.69 MiB | 28.14 MiB/s, done.
Resolving deltas: 100% (966/966), done.


In [74]:
cd avatarify

/content/avatarify/avatarify


In [75]:
# Instalar versões compatíveis com Colab
!pip install opencv-python-headless
!pip install face-alignment==1.0.0
!pip install pyzmq msgpack-numpy pyyaml requests
!pip install numpy torch torchvision

In [76]:
import sys
!{sys.executable} -m pip install --upgrade pip setuptools wheel
!git clone https://github.com/alievk/first-order-model.git fomm
!{sys.executable} -m pip install face-alignment==1.0.0 msgpack_numpy pyyaml==5.1

Cloning into 'fomm'...
remote: Enumerating objects: 211, done.
remote: Total 211 (delta 0), reused 0 (delta 0), pack-reused 211 (from 1)
Receiving objects: 100% (211/211), 58.16 MiB | 17.75 MiB/s, done.
Resolving deltas: 100% (108/108), done.


In [77]:
!avatarify/scripts/download_data.sh

/bin/bash: line 1: avatarify/scripts/download_data.sh: No such file or directory


In [78]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### ngrok
Follow the steps below to setup ngrok. You will also need to sign up on the ngrok site and get your authtoken (free).


In [79]:
# Download ngrok
!avatarify/scripts/get_ngrok.sh

/bin/bash: line 1: avatarify/scripts/get_ngrok.sh: No such file or directory


# Run
Start here if the runtime was restarted after installation.

In [80]:
cd /content/avatarify

/content/avatarify


In [81]:
#!git pull origin

In [88]:
from subprocess import Popen, PIPE
import shlex
import json
import time


def run_with_pipe(command):
  commands = list(map(shlex.split,command.split("|")))
  ps = Popen(commands[0], stdout=PIPE, stderr=PIPE)
  for command in commands[1:]:
    ps = Popen(command, stdin=ps.stdout, stdout=PIPE, stderr=PIPE)
  return ps.stdout.readlines()


def get_tunnel_adresses():
  info = run_with_pipe("curl http://localhost:4040/api/tunnels")
  assert info

  info = json.loads(info[0])
  for tunnel in info['tunnels']:
    url = tunnel['public_url']
    port = url.split(':')[-1]
    local_port = tunnel['config']['addr'].split(':')[-1]
    print(f'{url} -> {local_port} [{tunnel["name"]}]')
    if tunnel['name'] == 'input':
      in_addr = url
    elif tunnel['name'] == 'output':
      out_addr = url
    else:
      print(f'unknown tunnel: {tunnel["name"]}')

  return in_addr, out_addr

In [86]:
# Input and output ports for communication
local_in_port = 5557
local_out_port = 5558

# Start the worker


In [87]:
# (Re)Start the worker
with open('/tmp/run.txt', 'w') as f:
  ps = Popen(
      shlex.split(f'./run.sh --is-worker --in-port {local_in_port} --out-port {local_out_port} --no-vcam --no-conda'),
      stdout=f, stderr=f, cwd='/content/avatarify')
  time.sleep(3)

FileNotFoundError: [Errno 2] No such file or directory: './run.sh'

This command should print lines if the worker is successfully started

In [89]:
!ps aux | grep 'python3 afy/cam_fomm.py' | grep -v grep | tee /tmp/ps_run
!if [[ $(cat /tmp/ps_run | wc -l) == "0" ]]; then echo "Worker failed to start"; cat /tmp/run.txt; else echo "Worker started"; fi

root       13609  1.3  3.9 4486820 526700 ?      S    01:56   0:02 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       13622  0.0  2.3 4552356 311088 ?      Sl   01:56   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       13625  0.0  2.3 4486820 308840 ?      S    01:56   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       13626  0.0  2.3 4552356 311120 ?      Sl   01:56   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-po

In [90]:
print('Verificando o conteúdo da linha 82 do arquivo problematico:')
!sed -n '82p' /usr/local/lib/python3.12/dist-packages/face_alignment/utils.py

Verificando o conteúdo da linha 82 do arquivo problematico:
    t[0, 0] = t.new_tensor(resolution / h)


# Open ngrok tunnel

#### Get ngrok token
Go to https://dashboard.ngrok.com/auth/your-authtoken (sign up if required), copy your authtoken and put it below.

In [91]:
# Paste your authtoken here in quotes
authtoken = "2jzZdtnyHPlfh72h1yxqSUdPTIC_2mSfFUPaZy6FEVsDKbxXZ"

Set your region

Code | Region
--- | ---
us | United States
eu | Europe
ap | Asia/Pacific
au | Australia
sa | South America
jp | Japan
in | India

In [92]:
# Set your region here in quotes
region = "eu"

In [93]:
authtoken = "2jzZdtnyHPlfh72h1yxqSUdPTIC_2mSfFUPaZy6FEVsDKbxXZ"
region = "eu"
local_in_port = 5557
local_out_port = 5558
config =\
f"""
version: 2
authtoken: {authtoken}
region: {region}
console_ui: False
tunnels:
  input:
    addr: {local_in_port}
    proto: tcp
  output:
    addr: {local_out_port}
    proto: tcp
"""

with open('ngrok.conf', 'w') as f:
  f.write(config)

In [95]:
ps = Popen('./avatarify/scripts/open_tunnel_ngrok.sh', stdout=PIPE, stderr=PIPE)
time.sleep(3)

In [96]:
# Get tunnel addresses
try:
  in_addr, out_addr = get_tunnel_adresses()
  print("Tunnel opened")
except Exception as e:
  [print(l.decode(), end='') for l in ps.stdout.readlines()]
  print("Something went wrong, reopen the tunnel")

Opening tunnel
Something went wrong, reopen the tunnel


### [Optional] AWS proxy
Alternatively you can create a ssh reverse tunnel to an AWS `t3.micro` instance (it's free). It has lower latency than ngrok.

1. In your AWS console go to Services -> EC2 -> Instances -> Launch Instance;
1. Choose `Ubuntu Server 18.04 LTS` AMI;
1. Choose `t3.micro` instance type and press Review and launch;
1. Confirm your key pair and press Launch instances;
1. Go to the security group of this instance and edit inbound rules. Add TCP ports 5557 and 5558 and set Source to Anywhere. Press Save rules;
1. ssh into the instance (you can find the command in the Instances if you click on the Connect button) and add this line in the end of `/etc/ssh/sshd_config`:
```
GatewayPorts yes
```
then restart `sshd`
```
sudo service sshd restart
```
1. Copy your `key_pair.pem` by dragging and dropping it into avatarify folder in this notebook;
1. Use the command below to open the tunnel;
1. Start client with a command (substitute `run_mac.sh` with `run_windows.bat` or `run.sh`)
```
./run_mac.sh --is-client --in-addr tcp://instace.compute.amazonaws.com:5557 --out-addr tcp://instance.compute.amazonaws.com:5558
```

In [ ]:
# Open reverse ssh tunnel (uncomment line below)
# !./scripts/open_tunnel_ssh.sh key_pair.pem ubuntu@instance.compute.amazonaws.com

# Start the client
When you run the cell below it will print a command. Run this command on your computer:

1. Open a terminal (in Windows open `Anaconda Prompt`);
2. Change working directory to the `avatarify` directory:</br>
* Windows (change `C:\path\to\avatarify` to your path)</br>
`cd C:\path\to\avatarify`</br></br>
* Mac/Linux (change `/path/to/avatarify` to your path)</br>
`cd /path/to/avatarify`
3. Copy-paste to the terminal the command below and run;
4. It can take some time to connect (usually up to 10 seconds). If the preview window doesn't appear in a minute or two, look for the errors above in this notebook and report in the [issues](https://github.com/alievk/avatarify/issues) or [Slack](https://join.slack.com/t/avatarify/shared_invite/zt-dyoqy8tc-~4U2ObQ6WoxuwSaWKKVOgg).

In [ ]:
print('Copy-paste to the terminal the command below and run (press Enter)\n')
print('Mac:')
print(f'./run_mac.sh --is-client --in-addr {in_addr} --out-addr {out_addr}')
print('\nWindows:')
print(f'run_windows.bat --is-client --in-addr {in_addr} --out-addr {out_addr}')
print('\nLinux:')
print(f'./run.sh --is-client --in-addr {in_addr} --out-addr {out_addr}')

# Logs

If something doesn't work as expected, please run the cells below and include the logs in your report.

In [ ]:
#@title
!cat ./var/log/cam_fomm.log | head -100

In [ ]:
#@title
!cat ./var/log/recv_worker.log | tail -100

In [ ]:
#@title
!cat ./var/log/predictor_worker.log | tail -100

In [ ]:
#@title
!cat ./var/log/send_worker.log | tail -100

In [ ]:
!ls -la
!ls -la vox-adv-cpk.pth.tar
!ls -la fomm/config/vox-adv-256.yaml

In [ ]:
import sys
print("Python path:", sys.path)
try:
    import fomm
    print("fomm imported successfully")
except ImportError as e:
    print("fomm import error:", e)

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import torch
try:
    checkpoint = torch.load('vox-adv-cpk.pth.tar', map_location='cpu')
    print("Modelo carregado com sucesso")
    print("Chaves:", list(checkpoint.keys()))
except Exception as e:
    print("Erro ao carregar modelo:", e)

# Task
To fully diagnose the 'predictor_worker error', execute the following cell to retrieve the complete log from `predictor_worker.log`:

```python
#@title
!cat ./var/log/predictor_worker.log
```

In [ ]:
# No Colab, fazer upload dos arquivos
from google.colab import files
uploaded = files.upload()

In [ ]:
# Reinstalar dependências
!pip install -r requirements.txt
!pip install -e .

In [ ]:
print('Desinstalando face-alignment...')
!pip uninstall -y face-alignment

In [ ]:
print('Reinstalando face-alignment==1.0.0...')
!pip install face-alignment==1.0.0

In [ ]:
import fileinput
import sys
import os

file_path = '/usr/local/lib/python3.12/dist-packages/face_alignment/utils.py'
old_line = '    t[0, 0] = resolution / h'
new_line = '    t[0, 0] = t.new_tensor(resolution / h)'

if os.path.exists(file_path):
    print(f"Reaplicando patch em {file_path}...")
    with fileinput.FileInput(file_path, inplace=True, backup='.bak') as file:
        for line in file:
            if line.strip() == old_line.strip():
                print(new_line)
            else:
                print(line, end='')
    print("Patch reaplicado com sucesso.")
else:
    print(f"Erro: {file_path} não encontrado. Incapaz de reaplicar o patch.")

### Próximo passo:

Após a execução das células acima, por favor, **reinicie o worker** executando novamente a célula `8PnArK75mRqx` e, em seguida, verifique o log (`38ee77d2`) para confirmar que o erro foi resolvido.

## Get Full Predictor Worker Log

### Subtask:
Retrieve the complete log output from `predictor_worker.log` to diagnose the 'predictor_worker error'.


**Reasoning**:
The user explicitly asked to retrieve the complete log output from `predictor_worker.log`. The `cat` command will display the entire content of the specified file.



In [ ]:
print('Displaying full content of predictor_worker.log:')
!cat ./var/log/predictor_worker.log

In [ ]:
print('Displaying updated content of predictor_worker.log:')
!cat ./var/log/predictor_worker.log

In [ ]:
import fileinput
import sys
import os

file_path = '/usr/local/lib/python3.12/dist-packages/face_alignment/utils.py'
old_line = '    t[0, 0] = resolution / h'
new_line = '    t[0, 0] = t.new_tensor(resolution / h)'

if os.path.exists(file_path):
    print(f"Patching {file_path}...")
    with fileinput.FileInput(file_path, inplace=True, backup='.bak') as file:
        for line in file:
            if line.strip() == old_line.strip():
                print(new_line)
            else:
                print(line, end='')
    print("Patch applied successfully.")
    print("Please restart the worker by running cell 8PnArK75mRqx, and then re-check the logs if necessary.")
else:
    print(f"Error: {file_path} not found. Unable to apply patch.")


### Next Steps:
1.  Execute the cell above to apply the patch.
2.  Re-run the worker start cell (`8PnArK75mRqx`).
3.  Check the logs again (cells `W0eY8gkBqUJG` and the log-checking cells like `0GeT7KxON0Ke`, `YThWBXCf_yzI`, etc.) to confirm the error is resolved.

## Inspecting `/tmp/run.txt` for detailed error

The `run.sh` script's output, including any errors, is logged to `/tmp/run.txt`. Let's examine this file for more details about the `predictor_worker` error.

In [ ]:
print('Displaying full content of /tmp/run.txt:')
!cat /tmp/run.txt

## Final Task

### Subtask:
Review the full log to identify the root cause of the `predictor_worker` error and provide a solution or next steps.


## Summary:

### Q&A
The current step successfully retrieved the `predictor_worker.log` content. However, the root cause of the `predictor_worker error` has not yet been identified, nor has a solution or next steps been provided for resolving the error itself. This step was foundational for the subsequent diagnosis.

### Data Analysis Key Findings
*   The log file `predictor_worker.log` was successfully retrieved and displayed.
*   The log indicates that the predictor was initialized with specific configuration parameters.
*   An explicit error message, `predictor_worker error`, was recorded in the log.
*   Following the error, an entry `predictor_worker exit` was logged, suggesting an immediate termination of the worker process after the error.
*   Timestamps are available for each log entry, allowing for chronological analysis of events.

### Insights or Next Steps
*   Further analyze the `predictor_worker error` log entry and surrounding messages (if any) in the retrieved log to pinpoint the exact nature and context of the error.
*   Based on the identified error, investigate relevant configuration parameters, dependencies, or code sections that might lead to the `predictor_worker` failure and subsequent exit.
